# Интроспекция: inspect, code objects, dis

Как программа смотрит на саму себя. Нужно для домашних задач
`arg_binding` (свой `bind_args` **без** `inspect`) и `codeops` (счётчик
байткод-операций).

Примеры — на живых функциях `minipipe`, а не на `def f(): pass`.

| # | О чём |
|---|---|
| 1 | `inspect`: подпись, исходник, члены модуля |
| 2 | Стек и кадры |
| 3 | Code object: из чего он состоит |
| 4 | Собираем `bind_args` руками |
| 5 | `dis`: читаемый дамп |
| 6 | `dis.Bytecode`: считаем операции |
| 7 | Вложенные code objects |

In [ ]:
import dis
import inspect

import minipipe
from minipipe import accuracy, minmax_scale, train_test_split

print("minipipe", minipipe.__version__)

## 1. `inspect`

Подпись функции — объект, а не строка: по нему можно ходить.

In [ ]:
signature = inspect.signature(train_test_split)
print(signature, "\n")

for name, parameter in signature.parameters.items():
    default = "—" if parameter.default is inspect.Parameter.empty else repr(parameter.default)
    print(f"  {name:12} {parameter.kind.description:22} по умолчанию {default}")

Исходник тоже достаётся — если он есть на диске.

In [ ]:
print(inspect.getsource(minmax_scale))
print("файл:", inspect.getsourcefile(minmax_scale))
print("строка:", inspect.getsourcelines(minmax_scale)[1])

Члены модуля — с предикатом, чтобы отобрать нужное.

In [ ]:
functions = inspect.getmembers(minipipe.prep, inspect.isfunction)
print("функции в minipipe.prep:")
for name, function in functions:
    print(f"  {name:16} {inspect.signature(function)}")

print("\nклассы в minipipe:")
for name, cls in inspect.getmembers(minipipe, inspect.isclass):
    print(" ", name)

## 2. Стек и кадры

`inspect.stack()` отдаёт кадры вызовов — то же, что видит pdb командой
`where`. Полезно, когда надо узнать, кто тебя позвал.

In [ ]:
def who_called_me() -> None:
    for frame_info in inspect.stack()[1:4]:
        print(f"  {frame_info.function:20} строка {frame_info.lineno:4}  {frame_info.filename.split('/')[-1]}")


def middle() -> None:
    who_called_me()


def outer() -> None:
    middle()


print("цепочка вызовов, снизу вверх:")
outer()

In [ ]:
def show_locals() -> None:
    frame = inspect.currentframe()
    caller = frame.f_back if frame else None
    if caller is not None:
        print("локали вызывающего:", sorted(caller.f_locals))


def compute() -> None:
    rows = 10
    columns = 3
    show_locals()


compute()

Это тот же механизм, на котором стоит `pdb`: он ходит по `f_back` и
читает `f_locals`. Отсюда и ограничение из семинара — чего нет в кадре,
того не покажет никакой отладчик.

## 3. Code object

У каждой функции есть `__code__` — объект с тем, что компилятор о ней
знает. `inspect` читает именно его, и в `arg_binding` придётся делать то
же самое руками.

In [ ]:
def greet(name, greeting="привет", /, *args, loud=False, **kwargs):
    """Пример со всеми видами параметров."""


c = greet.__code__

print("co_varnames        ", c.co_varnames)
print("co_argcount        ", c.co_argcount, "— сколько можно передать позиционно")
print("co_posonlyargcount ", c.co_posonlyargcount, "— из них только позиционно")
print("co_kwonlyargcount  ", c.co_kwonlyargcount, "— только по имени")
print("co_nlocals         ", c.co_nlocals)
print()
print("__defaults__       ", greet.__defaults__)
print("__kwdefaults__     ", greet.__kwdefaults__)

Наличие `*args` и `**kwargs` лежит не в отдельном поле, а битами во
`co_flags`.

In [ ]:
CO_VARARGS = 0x04
CO_VARKEYWORDS = 0x08

for function in (greet, minmax_scale, train_test_split):
    flags = function.__code__.co_flags
    print(f"  {function.__name__:18} *args: {bool(flags & CO_VARARGS):5} "
          f"**kwargs: {bool(flags & CO_VARKEYWORDS)}")

Порядок в `co_varnames` фиксирован, и это главное, на чём держится
байндинг:

```
позиционные (включая positional-only) -> keyword-only -> *args -> **kwargs -> локальные
```

## 5. `dis`: читаемый дамп

Что интерпретатор реально выполняет.

In [ ]:
dis.dis(accuracy)

Полезно, когда непонятно, во что превращается запись. Сравним две формы
одного и того же:

In [ ]:
def with_plus(items):
    result = ""
    for item in items:
        result = result + item
    return result


def with_join(items):
    return "".join(items)


for function in (with_plus, with_join):
    ops = [instruction.opname for instruction in dis.get_instructions(function)]
    print(f"  {function.__name__:12} {len(ops):3} инструкций")

## 6. `dis.Bytecode`: считаем операции

`dis.dis` печатает, а `dis.Bytecode` и `dis.get_instructions` отдают
объекты — по ним можно считать. Это половина задачи `codeops`.

In [ ]:
from collections import Counter

counts = Counter(instruction.opname for instruction in dis.Bytecode(minmax_scale))
for opname, count in counts.most_common(8):
    print(f"  {opname:22} {count}")
print(f"  ... всего {sum(counts.values())} инструкций, {len(counts)} разных")

## 7. Вложенные code objects

Вот главная тонкость `codeops`. Тело функции, тело comprehension, тело
класса — это **отдельные** code objects, и лежат они в `co_consts`
внешнего. `dis.Bytecode` по умолчанию туда не заходит.

In [ ]:
import types

source = compile(
    """
def outer():
    def inner():
        return 42
    return [inner() for _ in range(3)]
""",
    "<demo>",
    "exec",
)

print("верхний уровень:", [i.opname for i in dis.get_instructions(source)][:6], "...")
print()
print("что лежит в co_consts:")
for const in source.co_consts:
    kind = "CODE OBJECT" if isinstance(const, types.CodeType) else type(const).__name__
    name = getattr(const, "co_name", const)
    print(f"  {kind:12} {name!r}")

Значит обходить надо рекурсивно:

In [ ]:
def count_all(code: types.CodeType) -> Counter:
    """Считает операции во всём дереве code objects."""
    counts = Counter(instruction.opname for instruction in dis.get_instructions(code))
    for const in code.co_consts:
        if isinstance(const, types.CodeType):
            counts += count_all(const)
    return counts


shallow = Counter(i.opname for i in dis.get_instructions(source))
deep = count_all(source)

print(f"только верхний уровень: {sum(shallow.values())} инструкций")
print(f"с вложенными:           {sum(deep.values())} инструкций")
print()
print("что появилось только при рекурсии:")
for opname in sorted(set(deep) - set(shallow)):
    print(f"  {opname:22} {deep[opname]}")

`RETURN_VALUE` на верхнем уровне один, а в дереве — по одному на каждое
тело. Ровно про это предупреждает условие `codeops`.

Дальше остаётся обвязка: принять `types.CodeType`, вернуть
`dict[str, int]`. Логика — выше.